# Notebook D: how do mice combine the grating with the block prior? (IBL behavior, CPU + internet, ~20-40 min)

**Question this answers.** In the IBL task a grating appears left or right at some contrast; in blocks the grating is on the right 80% or 20% of the time. We fit the same model we fit to LLaVA,

$$P(\text{right}) = \lambda_L + (1-\lambda_L-\lambda_R)\,\sigma\!\big(b + \beta s + \gamma \pi\big),\qquad \pi=\log\frac{p_{right}}{1-p_{right}}\ (\pm1.386),$$

with $s$ the signed contrast (right positive, in fractions 0 to 1) and $\lambda_{L,R}$ the lapse rates (mice make errors even at 100% contrast). It gives:

1. **Mouse $\gamma$, $\gamma/\beta$, and the boundary shift $2.77\gamma/\beta$** (contrast-equivalent of 0.2 vs 0.8 block). VLM reference values from our LLaVA run: $\gamma=0.277$ with the history-suppressed $\beta=6.3$ gives a shift of **12%** contrast; using the no-history $\beta=15$ gives **5%**.
2. **Form of the combination:** is the prior an additive shift (our VLM result, flat across contrast) or does it grow with contrast / affect lapses? (model $M_3$: $\gamma=\gamma_0+\gamma_1|s|$.)
3. **What prior does the mouse actually follow?** Fit with the true block prior *and* the recent-10-trials count (the only information our prompt gave the VLM), to see which the animal tracks.
4. **Conflict trials:** accuracy when the stimulus is on the side the block disfavors vs favors it.

**Before you run:** Settings -> turn **Internet on**; no GPU needed. Use *Save & Run All (Commit)*. If the IBL server search fails the notebook prints the exact error; send it to me.

*This is the comparison of derived quantities only (a ratio, a shape), never raw units.*

In [1]:
# ===================== CONFIG =====================
MOCK = False
MAX_SESSIONS = 150          # sessions to load (each is small). 150 sessions ~ 15-30 min
MIN_TRIALS_SUBJECT = 600    # biased-block trials needed per mouse to fit it
OUT = "/kaggle/working/D_out"
import os, sys, subprocess, time, math, zipfile, warnings
warnings.filterwarnings("ignore")
if not MOCK:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ONE-api"], check=False)
import numpy as np, pandas as pd
from scipy.optimize import minimize
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [3]:
# ===================== LOAD IBL TRIALS =====================
def tidy(df, subj, eid):
    """Return one row per trial with: side, signed contrast s, p_right (block), chose_right. Drops no-go and unbiased trials later."""
    cl, cr = np.asarray(df["contrastLeft"], float), np.asarray(df["contrastRight"], float)
    right = ~np.isnan(cr)                                  # stimulus on the right if contrastRight is defined
    c = np.where(right, cr, cl)
    fb, ch, pl = np.asarray(df["feedbackType"], float), np.asarray(df["choice"], float), np.asarray(df["probabilityLeft"], float)
    chose_right = np.where(fb == 1, right, ~right)         # correct -> chose the stimulus side; incorrect -> the other side (robust to wheel sign conventions)
    ok = (ch != 0) & np.isfinite(c) & np.isfinite(fb) & np.isfinite(pl) & (fb != 0)
    return pd.DataFrame(dict(subject=subj, eid=eid, trial=np.arange(len(c)), right=right, contrast=c, s=np.where(right, c, -c),
                             p_right=np.round(1 - pl, 2), chose_right=chose_right.astype(int), ok=ok))

if MOCK:
    rng = np.random.default_rng(0); parts = []
    for k in range(25):
        b0, beta, gam, lL, lR = rng.normal(0, .15), rng.uniform(6, 14), rng.uniform(0.15, 0.5), rng.uniform(.0, .12), rng.uniform(.0, .12)
        for ses in range(3):
            n = 700; blocks = np.repeat(rng.choice([0.2, 0.8], 20), 40)[:n]; blocks[:90] = 0.5
            right = rng.random(n) < blocks; c = rng.choice([0, .0625, .125, .25, .5, 1.], n); s = np.where(right, c, -c)
            pi = np.log(blocks / (1 - blocks)); p = lL + (1 - lL - lR) / (1 + np.exp(-(b0 + beta * s + gam * pi)))
            chose = (rng.random(n) < p).astype(int)
            parts.append(pd.DataFrame(dict(subject=f"m{k}", eid=f"m{k}_{ses}", trial=np.arange(n), right=right, contrast=c, s=s, p_right=blocks, chose_right=chose, ok=True)))
    TR = pd.concat(parts, ignore_index=True); print("MOCK trials", TR.shape)
else:
    from one.api import ONE
    IBL_USER = IBL_PASS = None                                      # or type them here, e.g. IBL_USER = "myname"
    try:
        from kaggle_secrets import UserSecretsClient
        _sc = UserSecretsClient(); IBL_USER, IBL_PASS = _sc.get_secret("IBL_USER"), _sc.get_secret("IBL_PASS")
    except Exception as e:
        print("no Kaggle secrets IBL_USER / IBL_PASS:", type(e).__name__)
    one, errs = None, []
    for u, p in [(IBL_USER, IBL_PASS), ("intbrainlab", "international")]:         # your own account first, old shared public account as fallback
        if not u: continue
        try:
            one = ONE(base_url="https://openalyx.internationalbrainlab.org", username=u, password=p, silent=True); print("connected as", u); break
        except Exception as e:
            errs.append(f"{u}: {type(e).__name__}: {str(e)[:120]}")
    assert one is not None, "could not log in to openalyx. Create a free account at https://openalyx.internationalbrainlab.org/signup, add Kaggle secrets IBL_USER and IBL_PASS (Add-ons -> Secrets), rerun. Errors: " + " | ".join(errs)
    eids = None
    for name, fn in [("project=brainwide", lambda: one.search(project="ibl_neuropixel_brainwide_01")),
                     ("protocol=ephysChoiceWorld", lambda: one.search(task_protocol="ephysChoiceWorld")),
                     ("dataset=trials", lambda: one.search(dataset="trials.table.pqt"))]:
        try:
            eids = list(fn()); print(f"search '{name}': {len(eids)} sessions")
            if eids: break
        except Exception as e:
            print(f"search '{name}' failed: {type(e).__name__}: {e}")
    assert eids, "no sessions found: paste the printed errors to Claude"
    rng = np.random.default_rng(0); eids = list(rng.permutation(eids))[:MAX_SESSIONS]
    parts, t0 = [], time.time()
    for i, eid in enumerate(eids):
        try:
            try: tr = one.load_object(eid, "trials", collection="alf")
            except Exception: tr = one.load_object(eid, "trials")
            subj = one.get_details(eid)["subject"]
            need = ["contrastLeft", "contrastRight", "probabilityLeft", "choice", "feedbackType"]
            parts.append(tidy(pd.DataFrame({k: np.asarray(tr[k]) for k in need}), subj, str(eid)))
        except Exception as e:
            print(f"  skip {str(eid)[:8]}: {type(e).__name__}: {str(e)[:80]}")
        if i % 10 == 0: print(f"  loaded {i+1}/{len(eids)} ({time.time()-t0:.0f}s)", flush=True)
    TR = pd.concat(parts, ignore_index=True); print("trials", TR.shape, "| subjects", TR.subject.nunique(), "| sessions", TR.eid.nunique())

TR.to_csv(f"{OUT}/ibl_trials_tidy.csv", index=False)
print("contrast levels:", sorted(TR.contrast.dropna().unique())[:10], "| block values:", sorted(TR.p_right.unique()))

connected as ponishbhatia


Alyx server returned a detail message: This query matches 7505 records. Rather than paging through them, use the ONE API to query downloaded cache tables locally instead of the database: https://int-brain-lab.github.io/ONE/. Paging large result sets places avoidable load on this database and is heavily rate limited.


search 'project=brainwide': 7505 sessions


(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/mrsicflogellab/Subjects/SWC_029/2020-02-13/001/alf/_ibl_trials.repNum.npy: 100%|██████████| 4.13k/4.13k [00:00<00:00, 18.7kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/mrsicflogellab/Subjects/SWC_029/2020-02-13/001/alf/_ibl_trials.table.pqt: 100%|██████████| 42.9k/42.9k [00:00<00:00, 133kB/s]


  loaded 1/150 (4s)


(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/danlab/Subjects/DY_008/2020-01-24/002/alf/_ibl_trials.table.pqt: 100%|██████████| 13.8k/13.8k [00:00<00:00, 65.8kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/danlab/Subjects/DY_008/2020-01-24/002/alf/_ibl_trials.repNum.npy: 100%|██████████| 608/608 [00:00<00:00, 2.41kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/hoferlab/Subjects/SWC_003/2019-06-19/001/alf/_ibl_trials.table.pqt: 100%|██████████| 24.6k/24.6k [00:00<00:00, 76.8kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/hoferlab/Subjects/SWC_003/2019-06-19/001/alf/_ibl_trials.repNum.npy: 100%|██████████| 1.90k/1.90k [00:00<00:00, 7.19kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/danlab/Subjects/DY_015/2020-02-07/001/alf/_ibl_trials.table.pqt: 100%|██████████| 24.1k/24.1k [00:00<00:00, 105kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/danlab/Subjects/DY_015/2020-02-07/001/alf/_ibl_

... (43 outputs left out)

(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/cortexlab/Subjects/KS014/2019-09-23/001/alf/_ibl_trials.table.pqt: 100%|██████████| 35.0k/35.0k [00:00<00:00, 109kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_05/2019-07-19/002/alf/_ibl_trials.table.pqt: 100%|██████████| 98.5k/98.5k [00:00<00:00, 195kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_14/2019-09-19/001/alf/_ibl_trials.table.pqt: 100%|██████████| 58.7k/58.7k [00:00<00:00, 184kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/mrsicflogellab/Subjects/SWC_023/2019-09-30/002/alf/_ibl_trials.table.pqt: 100%|██████████| 75.8k/75.8k [00:00<00:00, 166kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/danlab/Subjects/DY_009/2019-12-28/001/alf/_ibl_trials.repNum.npy: 100%|██████████| 4.70k/4.70k [00:00<00:00, 20.7kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/danlab/Subjects/DY_009

  loaded 141/150 (312s)



(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_07/2019-06-18/001/alf/_ibl_trials.table.pqt: 100%|██████████| 69.0k/69.0k [00:00<00:00, 173kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_16/2019-12-20/001/alf/_ibl_trials.table.pqt: 100%|██████████| 92.5k/92.5k [00:00<00:00, 204kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_17/2020-02-24/001/alf/_ibl_trials.repNum.npy: 100%|██████████| 7.42k/7.42k [00:00<00:00, 30.1kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_17/2020-02-24/001/alf/_ibl_trials.table.pqt: 100%|██████████| 70.6k/70.6k [00:00<00:00, 223kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wittenlab/Subjects/ibl_witten_15/2019-08-28/001/alf/_ibl_trials.table.pqt: 100%|██████████| 58.7k/58.7k [00:00<00:00, 165kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/wi

  skip 9b321fea: ALFObjectNotFound: trials 
 The ALF object was not found.  This may occur if the object or namespac
trials (91859, 9) | subjects 92 | sessions 139
contrast levels: [np.float64(0.0), np.float64(0.0625), np.float64(0.125), np.float64(0.25), np.float64(0.5), np.float64(1.0)] | block values: [np.float64(0.0), np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0)]


In [4]:
# ===================== recent-10 count regressor (what the VLM prompt contained) =====================
TR = TR.sort_values(["eid", "trial"]).reset_index(drop=True)
def recent_logodds(g, w=10):
    r = g.right.astype(float).values; out = np.full(len(r), np.nan)
    for i in range(w, len(r)):
        f = (r[i - w:i].sum() + 0.5) / (w + 1); out[i] = np.log(f / (1 - f))
    return pd.Series(out, index=g.index)
TR["tp"] = TR.groupby("eid", group_keys=False).apply(recent_logodds)
B = TR[TR.ok & TR.p_right.isin([0.2, 0.8]) & TR.tp.notna()].copy()
B["pi"] = np.log(B.p_right / (1 - B.p_right)); B["abs_s"] = B.s.abs()
cnt = B.groupby("subject").size(); keep = cnt[cnt >= MIN_TRIALS_SUBJECT].index
B = B[B.subject.isin(keep)]
print(f"biased-block trials {len(B)} | mice with >= {MIN_TRIALS_SUBJECT} trials: {len(keep)}")

biased-block trials 34941 | mice with >= 600 trials: 31


In [5]:
# ===================== model fits =====================
def sig(x): return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def make_nll(model, d):
    s, pi, a, tp, y = d.s.values, d.pi.values, d.abs_s.values, d.tp.values, d.chose_right.values
    def lin(p):
        if model == "M0": return p[0] + p[1] * s
        if model == "M1": return p[0] + p[1] * s + p[2] * pi
        if model == "M3": return p[0] + p[1] * s + (p[2] + p[5] * a) * pi
        if model == "M5": return p[0] + p[1] * s + p[2] * pi + p[5] * tp
        if model == "M6": return p[0] + p[1] * s + p[5] * tp                    # recent-10 only (no block prior)
    def f(p):
        pr = p[3] + (1 - p[3] - p[4]) * sig(lin(p)); pr = np.clip(pr, 1e-6, 1 - 1e-6)
        return -(y * np.log(pr) + (1 - y) * np.log(1 - pr)).sum()
    return f

def fit(model, d, starts=4, seed=0):
    f = make_nll(model, d); rng = np.random.default_rng(seed); best = None
    bounds = [(-3, 3), (0, 60), (-3, 3), (0, .5), (0, .5), (-10, 10)]
    for _ in range(starts):
        x0 = [rng.normal(0, .1), rng.uniform(5, 15), rng.uniform(0, .4), rng.uniform(.01, .1), rng.uniform(.01, .1), rng.normal(0, .1)]
        if model == "M0": x0[2] = 0; bnd = list(bounds); bnd[2] = (0, 1e-9)
        else: bnd = bounds
        r = minimize(f, x0, method="L-BFGS-B", bounds=bnd)
        if best is None or r.fun < best.fun: best = r
    return best

rows = []
for subj, d in B.groupby("subject"):
    rec = dict(subject=subj, n=len(d)); fits = {m: fit(m, d) for m in ["M0", "M1", "M3", "M5", "M6"]}
    p1 = fits["M1"].x; rec.update(b=p1[0], beta=p1[1], gamma=p1[2], lamL=p1[3], lamR=p1[4])
    rec["gamma_over_beta"] = p1[2] / p1[1] if p1[1] > 1e-6 else np.nan
    rec["shift_contrast"] = 2.7726 * rec["gamma_over_beta"]                      # boundary shift for 0.2 vs 0.8 blocks, in contrast fraction
    rec["c75"] = math.log(3) / p1[1] if p1[1] > 1e-6 else np.nan                  # contrast to move the logistic from 50% to 75%
    rec["shift_over_c75"] = rec["shift_contrast"] / rec["c75"] if rec["c75"] else np.nan
    rec["dAIC_prior"] = 2 * (fits["M0"].fun - fits["M1"].fun) - 2 * 1          # >0: prior term worth it (LR statistic - 2*params)
    rec["gamma1_contrast_dep"] = fits["M3"].x[5]; rec["dAIC_M3_vs_M1"] = 2 * (fits["M1"].fun - fits["M3"].fun) - 2
    rec["M5_gamma_block"], rec["M5_gamma_recent10"] = fits["M5"].x[2], fits["M5"].x[5]
    rec["dLL_block_vs_recent"] = fits["M6"].fun - fits["M1"].fun                   # >0: true block explains choices better than recent-10 alone
    rows.append(rec)
FIT = pd.DataFrame(rows); FIT.to_csv(f"{OUT}/mouse_fits.csv", index=False)
def med_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; rng = np.random.default_rng(seed)
    bs = [np.median(rng.choice(x, len(x))) for _ in range(n)]; return np.median(x), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
say(f"mice fitted: {len(FIT)} | median trials per mouse: {int(FIT.n.median())}")
for k, lab in [("beta", "beta (per unit contrast)"), ("gamma", "gamma (prior weight)"), ("gamma_over_beta", "gamma/beta"),
               ("shift_contrast", "boundary shift for 0.2 vs 0.8 (contrast fraction)"), ("shift_over_c75", "shift / c75"),
               ("lamL", "lapse L"), ("lamR", "lapse R")]:
    m_, lo, hi = med_ci(FIT[k]); say(f"  {lab:52s} median {m_:7.3f}  [95% CI over mice {lo:.3f}, {hi:.3f}]   IQR {FIT[k].quantile(.25):.3f}-{FIT[k].quantile(.75):.3f}")
say(f"prior term preferred (dAIC>0) in {(FIT.dAIC_prior > 0).mean()*100:.0f}% of mice")
m_, lo, hi = med_ci(FIT.gamma1_contrast_dep); say(f"M3 contrast dependence gamma1 (gamma = gamma0 + gamma1*|s|): median {m_:+.2f} [{lo:+.2f},{hi:+.2f}]; M3 preferred (dAIC>0) in {(FIT.dAIC_M3_vs_M1 > 0).mean()*100:.0f}% of mice")
m1, _, _ = med_ci(FIT.M5_gamma_block); m2, _, _ = med_ci(FIT.M5_gamma_recent10)
say(f"M5 both regressors: median gamma_block {m1:+.3f}, gamma_recent10 {m2:+.3f}; block-prior model beats recent-10-only in {(FIT.dLL_block_vs_recent > 0).mean()*100:.0f}% of mice")

mice fitted: 31 | median trials per mouse: 932
  beta (per unit contrast)                             median  16.102  [95% CI over mice 13.067, 19.179]   IQR 11.503-26.475
  gamma (prior weight)                                 median   0.611  [95% CI over mice 0.534, 0.760]   IQR 0.421-0.816
  gamma/beta                                           median   0.035  [95% CI over mice 0.019, 0.054]   IQR 0.017-0.064
  boundary shift for 0.2 vs 0.8 (contrast fraction)    median   0.096  [95% CI over mice 0.054, 0.150]   IQR 0.048-0.177
  shift / c75                                          median   1.543  [95% CI over mice 1.348, 1.917]   IQR 1.061-2.058
  lapse L                                              median   0.025  [95% CI over mice 0.014, 0.059]   IQR 0.009-0.104
  lapse R                                              median   0.054  [95% CI over mice 0.025, 0.086]   IQR 0.015-0.108
prior term preferred (dAIC>0) in 90% of mice
M3 contrast dependence gamma1 (gamma = gamma0 + gamma1*|s

In [6]:
# ===================== psychometric by block, conflict trials, plots =====================
levels = sorted(B.s.unique())
pm = B.groupby(["p_right", "s"]).chose_right.mean().unstack(0)
cons = B.assign(favored=np.where(B.p_right > .5, 1, 0), stim=B.right.astype(int))
cons["conflict"] = cons.favored != cons.stim; cons["correct"] = (cons.chose_right == cons.stim).astype(int)
cf = cons.groupby(["abs_s", "conflict"]).correct.mean().unstack()
z0 = cons[cons.abs_s == 0]; z0_fav = (z0.chose_right == z0.favored).mean() if len(z0) else np.nan
say(f"\nat 0% contrast, fraction of choices toward the block-favored side: {z0_fav:.3f} (0.5 = ignores block), n={len(z0)}")
say("accuracy by |contrast|: stimulus on the block-disfavored side (conflict) vs favored side (congruent)")
say(cf.rename(columns={True: "conflict", False: "congruent"}).round(3).to_string())
cf.to_csv(f"{OUT}/conflict_accuracy.csv"); pm.to_csv(f"{OUT}/psychometric_by_block.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for pb, col in [(0.2, "tab:blue"), (0.8, "tab:red")]: ax[0].plot(pm.index, pm[pb], "-o", c=col, label=f"block p_right={pb}")
ax[0].set_xlabel("signed contrast"); ax[0].set_ylabel("P(choose right)"); ax[0].set_title("mouse psychometric by block (pooled)"); ax[0].legend()
ax[1].hist(FIT.shift_contrast.clip(-.5, .5), bins=20); ax[1].axvline(.051, c="g", ls="--", label="LLaVA (beta none)"); ax[1].axvline(.122, c="r", ls="--", label="LLaVA (beta hist)")
ax[1].set_title("boundary shift for 0.2 vs 0.8 (contrast fraction), per mouse"); ax[1].legend()
ax[2].scatter(FIT.beta, FIT.gamma_over_beta, s=12); ax[2].set_xlabel("beta"); ax[2].set_ylabel("gamma/beta"); ax[2].set_title("per mouse")
plt.tight_layout(); plt.savefig(f"{OUT}/mouse_overview.png", dpi=130); plt.close()
if MOCK: say("MOCK check: true gamma in simulation ~U(0.15,0.5) (median ~0.33), beta ~U(6,14); the fits above should land near those.")
open(f"{OUT}/D_summary.txt", "w").write("\n".join(SUMMARY))
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"\nPaste everything printed in the previous two cells back to Claude. zip: {OUT}.zip")


at 0% contrast, fraction of choices toward the block-favored side: 0.665 (0.5 = ignores block), n=6246
accuracy by |contrast|: stimulus on the block-disfavored side (conflict) vs favored side (congruent)
conflict  congruent  conflict
abs_s                        
0.0000        0.658     0.306
0.0625        0.779     0.530
0.1250        0.863     0.697
0.2500        0.919     0.798
0.5000        0.254     0.810
1.0000        0.926     0.867

Paste everything printed in the previous two cells back to Claude. zip: /kaggle/working/D_out.zip
